# Testing Model Manual

---

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import timedelta

DOMESTIC_LOCS = {"Hanoi", "HCM City", "Da Nang", "Can Tho", "Hai Phong"}

MODEL_PATH = "../data/model_detection.json"
CSV_PATH   = "../data/ground_true_transactions_202607041510.csv"
THRESHOLD  = 0.9184

FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "log_amount_ratio_24h",
    "tx_count_1h", "tx_count_3h",
    "time_since_last_tx_sec",
]

model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

raw = pd.read_csv(CSV_PATH)
raw["event_time"] = pd.to_datetime(raw["event_time"], format='mixed')
raw = raw.sort_values(["account_id", "event_time"]).reset_index(drop=True)

home_map = raw.groupby("account_id")["location_name"].first().to_dict()

rows = []
for acc_id, grp in raw.groupby("account_id", sort=False):
    home    = home_map[acc_id]
    history = []

    for _, tx in grp.iterrows():
        t      = tx["event_time"]
        amount = float(tx["amount"])
        loc    = tx["location_name"]
        is_dom = int(loc in DOMESTIC_LOCS)

        cutoff_1h  = t - timedelta(hours=1)
        cutoff_3h  = t - timedelta(hours=3)
        cutoff_6h  = t - timedelta(hours=6)
        cutoff_24h = t - timedelta(hours=24)
        cutoff_1w  = t - timedelta(days=7)

        prev_24h   = [(pt, pa) for pt, pa, _d in history if pt >= cutoff_24h]
        avg_24h    = sum(pa for _, pa in prev_24h) / len(prev_24h) if prev_24h else amount
        cnt_1h     = sum(1 for pt, _a, _d in history if pt >= cutoff_1h)
        cnt_3h     = sum(1 for pt, _a, _d in history if pt >= cutoff_3h)
        last_ts    = max((pt for pt, _a, _d in history), default=None)
        time_since = (t - last_ts).total_seconds() if last_ts else 0.0

        rows.append({
            "transaction_id":         tx["transaction_id"],
            "account_id":             acc_id,
            "event_time":             t,
            "location_name":          loc,
            "is_fraud_label":         tx["is_fraud"],
            "fraud_pattern":          tx["fraud_pattern"],
            "amount":                 amount,
            "hour_of_day":            t.hour,
            "is_home":                int(loc == home),
            "is_domestic":            is_dom,
            "avg_amount_24h":         round(avg_24h, 2),
            "log_amount_ratio_24h":   round(np.log1p(amount / avg_24h), 6),
            "tx_count_1h":            cnt_1h,
            "tx_count_3h":            cnt_3h,
            "time_since_last_tx_sec": round(time_since, 1),
        })
        history.append((t, amount, is_dom))

feat_df = pd.DataFrame(rows)

X      = feat_df[FEATURES].astype(float)
dmat   = xgb.DMatrix(X, feature_names=FEATURES)
scores = model.get_booster().predict(dmat)

feat_df["risk_score"] = scores
feat_df["predicted"]  = scores >= THRESHOLD

cols = ["transaction_id", "account_id", "event_time", "location_name",
        "amount", "avg_amount_24h", "log_amount_ratio_24h",
        "tx_count_1h", "tx_count_3h", 
        "time_since_last_tx_sec",
        "is_home", "is_domestic", "hour_of_day",
        "risk_score", "predicted", "is_fraud_label", "fraud_pattern"]

feat_df[cols]

,transaction_id,account_id,event_time,location_name,amount,avg_amount_24h,log_amount_ratio_24h,tx_count_1h,tx_count_3h,time_since_last_tx_sec,is_home,is_domestic,hour_of_day,risk_score,predicted,is_fraud_label,fraud_pattern
0,dbc5e9ef-8982-4ba0-9085-f853c03e61ab,ACC_00042,2026-06-24 17:45:59.112423,Hai Phong,901000.0,901000.0,0.693147,0,0,0.0,1,1,17,1.621282e-06,False,False,NONE
1,15ac1d24-1273-4a4f-9fc5-a25b89a2256b,ACC_00042,2026-06-24 19:13:07.917838,Hai Phong,267934874.0,901000.0,5.698351,0,1,5228.8,1,1,19,6.142774e-03,False,False,ADVANCED_HIGH_FREQUENCY_V2
2,e343ac54-da5b-4d6a-9742-42e1b4d1d0e2,ACC_00042,2026-06-24 19:24:49.047308,Hai Phong,299537994.0,134417937.0,1.171989,1,2,701.1,1,1,19,9.971398e-01,True,True,ADVANCED_HIGH_FREQUENCY_V2
3,6d885972-af45-4ebf-be33-a64003866105,ACC_00042,2026-06-24 20:18:59.608923,Hai Phong,280113736.0,189457956.0,0.907654,1,3,3250.6,1,1,20,7.052712e-01,False,True,ADVANCED_HIGH_FREQUENCY_V2
4,05eb6104-0654-4c70-83ab-b716ce52473b,ACC_00042,2026-06-24 20:21:49.180614,Hai Phong,91145997.0,212121901.0,0.357455,2,4,169.6,1,1,20,9.997935e-01,True,True,ADVANCED_HIGH_FREQUENCY_V2
5,9d3c52e1-28a1-4270-9a6a-8e3427413846,ACC_00042,2026-06-26 00:02:02.564055,Hai Phong,907000.0,907000.0,0.693147,0,0,99613.4,1,1,0,1.730502e-06,False,False,NONE
6,692c849e-5706-48f4-ad32-1e33d897f961,ACC_00042,2026-06-28 20:28:47.912245,Hai Phong,759000.0,759000.0,0.693147,0,0,246405.3,1,1,20,9.300547e-07,False,False,NONE
7,8eff8c40-c55d-4f09-a3ef-3c2468daeb92,ACC_00042,2026-06-28 21:16:24.051371,Hai Phong,829000.0,759000.0,0.738229,1,1,2856.1,1,1,21,1.043931e-06,False,False,NONE
8,b7423c0e-08a4-4f31-85a2-5329b3b6c749,ACC_00042,2026-06-29 13:09:13.239484,Hai Phong,681000.0,794000.0,0.619330,0,0,57169.2,1,1,13,1.028467e-06,False,False,NONE
9,1fb6f4cf-6603-4860-8ee7-b6cad0c1f869,ACC_00042,2026-06-30 06:01:31.764662,Hai Phong,563000.0,681000.0,0.602525,0,0,60738.5,1,1,6,6.289623e-07,False,False,NONE


In [117]:
fn = feat_df[(feat_df["predicted"] == False) & (feat_df["is_fraud_label"] == True)]
fp = feat_df[(feat_df["predicted"] == True)  & (feat_df["is_fraud_label"] == False)]

print(f"False Negatives (missed fraud): {len(fn)}")
print(f"False Positives (false alarm):  {len(fp)}")

False Negatives (missed fraud): 46
False Positives (false alarm):  0


In [114]:
fn = feat_df[(feat_df["predicted"] == False) & (feat_df["is_fraud_label"] == True)]
print(fn.groupby("fraud_pattern").size().sort_values(ascending=False))

fraud_pattern
ADVANCED_HIGH_FREQUENCY_V2    70
ADVANCED_HIGH_FREQUENCY_V1    17
UNDER_THREAD_ACTIVITY          9
DECLINED_BURST                 4
HIGH_FREQUENCY                 2
LOCATION_JUMP                  1
dtype: int64


In [ ]:
fn_accounts = feat_df[(feat_df["predicted"] == False) & (feat_df["is_fraud_label"] == True)]["account_id"].unique()

fn_full = feat_df[feat_df["account_id"].isin(fn_accounts)].sort_values(["account_id", "event_time"])

cols = ["transaction_id", "account_id", "event_time", "location_name",
        "amount", "avg_amount_24h", "log_amount_ratio_24h",
        "tx_count_1h", "tx_count_3h",
        "time_since_last_tx_sec",
        "is_home", "is_domestic", "hour_of_day",
        "risk_score", "predicted", "is_fraud_label", "fraud_pattern"]

fn_full[cols].to_csv("../data/False Negatives.csv", index=False)
print(f"False Negative accounts : {len(fn_accounts)}")
print(f"Saved {len(fn_full)} rows → data/False Negatives JUMP LOCATION.csv")
fn_full[cols]

False Negative accounts : 35
Saved 248 rows → data/False Negatives JUMP LOCATION.csv


,transaction_id,account_id,event_time,location_name,amount,avg_amount_24h,log_amount_ratio_24h,avg_amount_1w,log_amount_ratio_1w,tx_count_1h,tx_count_3h,tx_count_6h,time_since_last_tx_sec,is_home,is_domestic,hour_of_day,risk_score,predicted,is_fraud_label,fraud_pattern
0,45c602fb-be72-4bf2-a30c-ea33c028476d,ACC_00112,2026-06-26 10:20:34.022138,Hai Phong,2719000.0,2.719000e+06,0.693147,2.719000e+06,0.693147,0,0,0,0.0,1,1,10,2.813903e-06,False,False,NONE
1,fe3d4174-bbac-4596-9f0b-6793a08869d5,ACC_00112,2026-06-27 04:15:58.176407,Hai Phong,883153.0,2.719000e+06,0.281268,2.719000e+06,0.281268,0,0,0,64524.2,1,1,4,4.333489e-08,False,False,NONE
2,5a34cbde-a262-49ac-9286-af7ea7eab56a,ACC_00112,2026-06-27 06:07:24.934036,HCM City,307151256.0,1.801076e+06,5.144802,1.801076e+06,5.144802,0,1,1,6686.8,0,1,6,9.925514e-03,False,True,ADVANCED_LOCATION_JUMP
3,9f72b3c2-26c9-4e84-ac04-b110145d16f5,ACC_00112,2026-06-27 06:24:53.648970,HCM City,274850629.0,1.035845e+08,1.295657,1.035845e+08,1.295657,1,2,2,1048.7,0,1,6,9.992836e-01,True,True,ADVANCED_LOCATION_JUMP
4,7dd686b4-cecb-4cb6-b78b-003392abb325,ACC_00112,2026-06-27 06:30:14.728189,HCM City,96870740.0,1.464010e+08,0.507830,1.464010e+08,0.507830,2,3,3,321.1,0,1,6,9.999310e-01,True,True,ADVANCED_LOCATION_JUMP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,fbd7b7c3-414c-4e78-abaf-a3d922eab13b,ACC_98942,2026-06-29 09:00:20.908548,Hai Phong,1817909.0,1.584000e+06,0.764383,1.584000e+06,0.764383,0,0,0,130675.4,1,1,9,6.887444e-09,False,False,NONE
360,b6a6133c-04a2-4984-b646-f7191036d258,ACC_98942,2026-06-29 09:09:01.116972,Hai Phong,669165.0,1.817909e+06,0.313420,1.700954e+06,0.331751,1,1,1,520.2,1,1,9,5.809012e-08,False,False,NONE
361,81acb4af-88ec-490a-ac79-a2dd2688ab16,ACC_98942,2026-06-29 10:44:57.496807,London,259783750.0,1.243537e+06,5.346665,1.357025e+06,5.259765,0,2,2,5756.4,0,0,10,1.814706e-01,False,True,ADVANCED_LOCATION_JUMP
362,24abc175-1711-4247-89cd-522f76c02c3e,ACC_98942,2026-06-29 10:47:21.638406,London,489655260.0,8.742361e+07,1.887214,6.596371e+07,2.130978,1,3,3,144.1,0,0,10,9.959846e-01,True,True,ADVANCED_LOCATION_JUMP
